In [ ]:
import pandas as pd
from fuzzywuzzy import fuzz
import numpy as np
from collections import defaultdict
from preprocess import preprocess_dataframe
import time

In [ ]:
# Măsoară timpul
start_time = time.time()

# Încarcă și preprocesează datele
df = pd.read_parquet('../veridion_entity_resolution_challenge.snappy.parquet', engine='pyarrow')
df = preprocess_dataframe(df)

print(f"Încărcare și preprocesare: {time.time() - start_time:.2f} secunde")

In [ ]:
# Resetăm contorul de timp
start_time = time.time()

# Folosim un blocking mai eficient cu mai multe chei
buckets = defaultdict(list)

# Blocking pe domeniu complet (mult mai specific decât doar prima literă)
for idx, row in df.iterrows():
    domain = str(row['website_domain']).lower() if not pd.isna(row['website_domain']) else ""
    if domain:
        buckets[f"domain_{domain}"].append(idx)

    # Blocking pe prima parte a adresei de email
    email = str(row['primary_email']).lower() if not pd.isna(row['primary_email']) else ""
    if email and '@' in email:
        email_prefix = email.split('@')[0]
        buckets[f"email_{email_prefix}"].append(idx)

    # Blocking pe prefixul numelui companiei (primele 3 caractere)
    company = str(row['company_name']).lower() if not pd.isna(row['company_name']) else ""
    if len(company) >= 3:
        buckets[f"name_{company[:3]}"].append(idx)

print(f"Număr de bucket-uri: {len(buckets)}")
print(f"Timp pentru creare bucket-uri: {time.time() - start_time:.2f} secunde")

# Măsoară dimensiunea medie și maximă a bucket-urilor
bucket_sizes = [len(b) for b in buckets.values()]
print(f"Dimensiune medie bucket: {np.mean(bucket_sizes):.2f}")
print(f"Dimensiune maximă bucket: {np.max(bucket_sizes)}")

In [ ]:
# Adaugă un cache pentru funcția de similaritate
def similarity_score(row1, row2):
    # Folosește un cache static în funcție
    if not hasattr(similarity_score, 'cache'):
        similarity_score.cache = {}
    
    # Folosește ID-urile rândurilor ca și cheie de cache
    cache_key = tuple(sorted([row1.name, row2.name]))
    
    if cache_key in similarity_score.cache:
        return similarity_score.cache[cache_key]
    
    # Restul funcției rămâne la fel...
    score = 0
    # ...
    
    # Salvează rezultatul în cache
    similarity_score.cache[cache_key] = score
    return score

In [ ]:
# Resetăm contorul de timp
threshold = 60  # prag de similaritate
start_time = time.time()

# Inițializăm structura de date pentru rezultate
visited = set()
groups = []

# O structură care ține evidența grupurilor
group_map = {}  # idx -> group_id

# Procesare bucket cu bucket
bucket_count = 0
for key, bucket in buckets.items():
    bucket_count += 1
    if bucket_count % 100 == 0:
        print(f"Procesăm bucket-ul {bucket_count}/{len(buckets)}")

    # Sortăm bucket-ul pentru a face procesarea mai eficientă
    bucket.sort()

    for i in range(len(bucket)):
        idx_i = bucket[i]

        # Dacă acest index e deja într-un grup, continuăm
        if idx_i in visited:
            continue

        # Verificăm dacă acest index e deja parte dintr-un grup
        if idx_i in group_map:
            group_id = group_map[idx_i]
        else:
            # Creăm un grup nou
            group_id = len(groups)
            groups.append([idx_i])
            group_map[idx_i] = group_id
            visited.add(idx_i)

        # Trecem prin restul bucket-ului pentru comparații
        for j in range(i + 1, len(bucket)):
            idx_j = bucket[j]

            # Dacă acest index e deja în grup, continuăm
            if idx_j in visited and group_map.get(idx_j) == group_id:
                continue

            # Calculăm scorul de similaritate
            score = similarity_score(df.loc[idx_i], df.loc[idx_j])

            if score >= threshold:
                # Dacă idx_j e deja într-un alt grup, unim grupurile
                if idx_j in group_map:
                    old_group_id = group_map[idx_j]
                    if old_group_id != group_id:
                        # Unim grupurile
                        for old_idx in groups[old_group_id]:
                            group_map[old_idx] = group_id
                        groups[group_